# Lane Detection — UFLD (Ultra Fast Lane Detection)

## 목표
- UFLD (ECCV 2020) 핵심 아이디어 **row-anchor classification** 직접 구현
- TuSimple 포맷 합성 데이터 생성 → 학습 → 평가
- TuSimple 공식 메트릭 (Accuracy / FP / FN) 파이프라인 구축

## 핵심 논문
- **Ultra Fast Structure-aware Deep Lane Detection** (Qin et al., ECCV 2020)
- 핵심 아이디어: 차선 검출을 **세그멘테이션이 아니라 행(row) 단위 분류 문제**로 변환
  - 이미지를 수평으로 N등분 → 각 행(row anchor)에서 차선이 몇 번째 열(column)에 있는가?
  - 세그멘테이션 대비 18배 빠름 (300+ FPS)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import json
import os
import random
import math
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. UFLD 아키텍처 이론

### Row-Anchor Classification 방식

```
입력 이미지 (H x W)
       ↓
ResNet Backbone (ImageNet pretrained)
       ↓
Feature Map → Global Average Pooling
       ↓
FC Head: (batch, num_lanes × num_row_anchors × (num_col_cells + 1))
       ↓
각 lane × 각 row anchor에서 → 몇 번째 column cell에 차선이 있는가?
       (+1 = 해당 행에 차선 없음 class)
```

### 파라미터
- `num_lanes` = 4 (최대 차선 수)
- `num_row_anchors` = 56 (행 기준점 개수)
- `num_col_cells` = 100 (열 분할 개수)
- 출력 shape: `(B, 4, 56, 101)` — 마지막 101 = 100 col + 1 no-lane

### 손실 함수
1. **분류 손실**: CrossEntropy — 각 row anchor에서 올바른 column 예측
2. **구조적 손실**: 인접 row anchor 간 예측값이 부드럽게 연결되도록 강제
   - `L_str = Σ (pred[i] - pred[i-1])^2`

In [ ]:
# ────────────────────────────────────────
# UFLD 하이퍼파라미터
# ────────────────────────────────────────
IMG_H, IMG_W = 288, 800          # UFLD 표준 입력 크기
NUM_LANES      = 4               # 최대 차선 수
NUM_ROW_ANCHORS = 56             # 수평 기준선 개수
NUM_COL_CELLS  = 100             # 열 분할 개수 (+ 1 no-lane)

# row anchor 위치 (이미지 하단 ~ 상단, 비율)
ROW_ANCHOR_RATIOS = np.linspace(0.42, 1.0, NUM_ROW_ANCHORS)  # 이미지 42%~100% 높이
ROW_ANCHORS_PX = (ROW_ANCHOR_RATIOS * IMG_H).astype(int)      # 픽셀 좌표

print(f'입력 크기: {IMG_H} x {IMG_W}')
print(f'Row anchors: {NUM_ROW_ANCHORS}개, y={ROW_ANCHORS_PX[0]}px ~ {ROW_ANCHORS_PX[-1]}px')
print(f'Column cells: {NUM_COL_CELLS}개')
print(f'출력 shape: (batch, {NUM_LANES}, {NUM_ROW_ANCHORS}, {NUM_COL_CELLS+1})')
print(f'출력 파라미터 수: {NUM_LANES * NUM_ROW_ANCHORS * (NUM_COL_CELLS+1):,}')

## 2. 합성 차선 데이터셋 생성

TuSimple 포맷으로 고속도로 차선 패턴을 수학적으로 생성합니다.
실제 TuSimple 구조와 동일한 JSON 레이블 형식을 사용합니다.

In [ ]:
def generate_lane_image(img_h=IMG_H, img_w=IMG_W, num_lanes=4, seed=None):
    """
    합성 차선 이미지 + TuSimple 포맷 레이블 생성
    Returns: PIL Image, h_samples(list), lanes(list of list)
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    img = Image.new('RGB', (img_w, img_h), color=(60, 60, 60))
    draw = ImageDraw.Draw(img)

    # 도로 배경 (어두운 회색)
    road_left  = int(img_w * 0.05)
    road_right = int(img_w * 0.95)
    draw.polygon([
        (road_left, img_h), (road_right, img_h),
        (int(img_w*0.65), int(img_h*0.42)),
        (int(img_w*0.35), int(img_h*0.42))
    ], fill=(80, 80, 80))

    # Vanishing point (소실점)
    vp_x = img_w / 2 + random.uniform(-30, 30)
    vp_y = img_h * 0.42

    h_samples = ROW_ANCHORS_PX.tolist()

    # 4개 차선 기준 x 위치 (하단 기준)
    lane_bottoms = [
        img_w * 0.20 + random.uniform(-20, 20),
        img_w * 0.40 + random.uniform(-15, 15),
        img_w * 0.60 + random.uniform(-15, 15),
        img_w * 0.80 + random.uniform(-20, 20),
    ]

    lanes = []
    colors = [(255, 255, 255), (255, 200, 0), (255, 200, 0), (255, 255, 255)]

    for li, bot_x in enumerate(lane_bottoms):
        lane_pts = []
        prev_pt = None
        for hy in h_samples:
            # 소실점 방향 선형 보간
            t = 1.0 - (hy - vp_y) / (img_h - vp_y + 1e-6)
            t = max(0.0, min(1.0, t))
            x = vp_x + (bot_x - vp_x) * (1.0 - t)

            # 곡률 노이즈
            curve = random.uniform(-1.5, 1.5) * (1 - t)
            x += curve

            if 0 <= x < img_w:
                lane_pts.append(int(x))
                if prev_pt is not None:
                    draw.line([prev_pt, (int(x), hy)], fill=colors[li], width=4)
                prev_pt = (int(x), hy)
            else:
                lane_pts.append(-2)   # TuSimple: -2 = 해당 row 없음
                prev_pt = None
        lanes.append(lane_pts)

    # 노이즈 추가 (현실감)
    img_arr = np.array(img)
    noise = np.random.randint(0, 25, img_arr.shape, dtype=np.uint8)
    img_arr = np.clip(img_arr.astype(int) + noise - 12, 0, 255).astype(np.uint8)
    img = Image.fromarray(img_arr)

    return img, h_samples, lanes


# 샘플 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    img, h_samples, lanes = generate_lane_image(seed=i*10)
    img_arr = np.array(img)
    ax.imshow(img_arr)
    for lane in lanes:
        xs = [x for x, h in zip(lane, h_samples) if x != -2]
        ys = [h for x, h in zip(lane, h_samples) if x != -2]
        if xs:
            ax.scatter(xs, ys, s=4, c='lime', zorder=5)
    ax.set_title(f'샘플 {i+1}: 합성 차선 (TuSimple 포맷)')
    ax.axis('off')
plt.suptitle('합성 차선 데이터셋 — Row Anchor 시각화', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('synthetic_lane_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print('샘플 이미지 저장: synthetic_lane_samples.png')

In [ ]:
class SyntheticLaneDataset(Dataset):
    """
    TuSimple 포맷 합성 차선 데이터셋
    - 매 __getitem__ 호출마다 새 이미지를 생성 (무한 데이터)
    - 레이블: (num_lanes, num_row_anchors) 형태의 column cell 인덱스
    """

    def __init__(self, size=2000, img_h=IMG_H, img_w=IMG_W,
                 num_col_cells=NUM_COL_CELLS, augment=True, seed_offset=0):
        self.size = size
        self.img_h = img_h
        self.img_w = img_w
        self.num_col_cells = num_col_cells
        self.augment = augment
        self.seed_offset = seed_offset

        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return self.size

    def _x_to_col(self, x):
        """픽셀 x 좌표 → column cell 인덱스 (0~num_col_cells-1, 없으면 num_col_cells)"""
        if x == -2:
            return self.num_col_cells   # no-lane class
        col = int(x / self.img_w * self.num_col_cells)
        return max(0, min(col, self.num_col_cells - 1))

    def __getitem__(self, idx):
        seed = idx + self.seed_offset
        img, h_samples, lanes = generate_lane_image(
            self.img_h, self.img_w, seed=seed
        )

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                img = T.functional.adjust_brightness(img, random.uniform(0.7, 1.3))
            if random.random() > 0.5:
                img = T.functional.adjust_contrast(img, random.uniform(0.8, 1.2))

        img_tensor = self.transform(img)

        # 레이블: (num_lanes, num_row_anchors) — column cell 인덱스
        label = torch.zeros(NUM_LANES, NUM_ROW_ANCHORS, dtype=torch.long)
        for li, lane in enumerate(lanes):
            for ri, x in enumerate(lane):
                label[li, ri] = self._x_to_col(x)

        return img_tensor, label


# 데이터셋 생성
train_dataset = SyntheticLaneDataset(size=2000, augment=True,  seed_offset=0)
val_dataset   = SyntheticLaneDataset(size=400,  augment=False, seed_offset=100000)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train: {len(train_dataset)}개, Val: {len(val_dataset)}개')
imgs, labels = next(iter(train_loader))
print(f'이미지 shape: {imgs.shape}')    # (16, 3, 288, 800)
print(f'레이블 shape: {labels.shape}')  # (16, 4, 56)
print(f'레이블 범위: {labels.min().item()} ~ {labels.max().item()} (no-lane={NUM_COL_CELLS})')

## 3. UFLD 모델 직접 구현

In [ ]:
class UFLD(nn.Module):
    """
    Ultra Fast Lane Detection (ECCV 2020)

    핵심 구조:
    1. ResNet-18 backbone (ImageNet 사전학습)
    2. Feature Pyramid: 마지막 3개 layer feature concat
    3. FC head: (num_lanes × num_row_anchors × (num_col_cells+1)) logits
    """

    def __init__(self,
                 num_lanes=NUM_LANES,
                 num_row_anchors=NUM_ROW_ANCHORS,
                 num_col_cells=NUM_COL_CELLS,
                 backbone='resnet18',
                 pretrained=True):
        super().__init__()

        self.num_lanes = num_lanes
        self.num_row_anchors = num_row_anchors
        self.num_col_cells = num_col_cells

        # ── Backbone ──────────────────────────────────
        resnet = models.resnet18(weights='DEFAULT' if pretrained else None)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1   # stride 4,  ch=64
        self.layer2 = resnet.layer2   # stride 8,  ch=128
        self.layer3 = resnet.layer3   # stride 16, ch=256
        self.layer4 = resnet.layer4   # stride 32, ch=512

        # ── Feature Aggregation ───────────────────────
        # layer2(128) + layer3(256) + layer4(512) → 896ch → 압축
        self.agg = nn.Sequential(
            nn.AdaptiveAvgPool2d((8, 22)),   # 공간 크기 통일
        )
        feat_dim = (128 + 256 + 512) * 8 * 22

        # ── Classifier Head ───────────────────────────
        out_dim = num_lanes * num_row_anchors * (num_col_cells + 1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(feat_dim, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, out_dim)
        )

    def forward(self, x):
        B = x.size(0)

        # Backbone
        x = self.layer0(x)
        x = self.layer1(x)
        f2 = self.layer2(x)   # 128ch
        f3 = self.layer3(f2)  # 256ch
        f4 = self.layer4(f3)  # 512ch

        # Feature pyramid 합치기
        agg2 = self.agg(f2)   # (B,128,8,22)
        agg3 = self.agg(f3)   # (B,256,8,22)
        agg4 = self.agg(f4)   # (B,512,8,22)
        feat = torch.cat([agg2, agg3, agg4], dim=1)  # (B,896,8,22)
        feat = feat.view(B, -1)                        # flatten

        # 분류
        out = self.classifier(feat)  # (B, num_lanes*num_row_anchors*(num_col_cells+1))
        out = out.view(B, self.num_lanes, self.num_row_anchors, self.num_col_cells + 1)
        return out


# 모델 생성 및 확인
model = UFLD(pretrained=True).to(device)

dummy = torch.randn(2, 3, IMG_H, IMG_W).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'출력 shape: {out.shape}')  # (2, 4, 56, 101)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total_params:,} ({total_params/1e6:.1f}M)')
print(f'학습 가능 파라미터: {trainable_params:,}')

## 4. 손실 함수 구현

### UFLD 손실 = 분류 손실 + 구조적 손실

**구조적 손실 (Structural Loss)**:
- 인접한 row anchor 간 예측 column이 갑자기 튀면 패널티
- 차선은 연속적이어야 한다는 사전 지식을 학습에 주입
- `argmax(pred[i]) - argmax(pred[i-1])` 의 제곱합 최소화

In [ ]:
class UFLDLoss(nn.Module):
    """
    UFLD 복합 손실 함수

    L_total = L_cls + lambda_str * L_str

    L_cls: CrossEntropy (no-lane class 포함)
    L_str: 인접 row anchor 간 부드러움 제약
    """

    def __init__(self, num_col_cells=NUM_COL_CELLS, lambda_str=1.0):
        super().__init__()
        self.num_col_cells = num_col_cells
        self.lambda_str    = lambda_str

        # no-lane(마지막 class)에 낮은 가중치 — 차선 없는 경우가 많아 불균형 보정
        weight = torch.ones(num_col_cells + 1)
        weight[-1] = 0.4   # no-lane class weight
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def forward(self, pred, target):
        """
        pred:   (B, num_lanes, num_row_anchors, num_col_cells+1)
        target: (B, num_lanes, num_row_anchors) — column cell 인덱스
        """
        B, L, R, C = pred.shape

        # ── 분류 손실 ──────────────────────────────────
        # reshape: (B*L*R, C) vs (B*L*R,)
        pred_flat   = pred.view(-1, C)
        target_flat = target.view(-1)
        l_cls = self.ce(pred_flat, target_flat)

        # ── 구조적 손실 ────────────────────────────────
        # no-lane을 제외한 유효 예측에서 softmax expectation으로 연속성 계산
        # pred shape: (B, L, R, C)
        # col_idx: column 인덱스 텐서 (0~num_col_cells-1)
        valid_pred = pred[:, :, :, :-1]  # no-lane class 제거 → (B,L,R,C-1)
        col_idx = torch.arange(self.num_col_cells, dtype=torch.float32, device=pred.device)

        # Softmax expectation: 각 row의 예상 column 위치
        softmax_prob = F.softmax(valid_pred, dim=-1)          # (B,L,R,C-1)
        expected_col = (softmax_prob * col_idx).sum(dim=-1)   # (B,L,R)

        # 인접 row 간 차이 (절댓값 합)
        diff = expected_col[:, :, 1:] - expected_col[:, :, :-1]  # (B,L,R-1)
        l_str = diff.pow(2).mean()

        loss = l_cls + self.lambda_str * l_str
        return loss, l_cls.item(), l_str.item()


criterion = UFLDLoss(lambda_str=0.5).to(device)

# 손실 함수 테스트
dummy_pred   = torch.randn(4, NUM_LANES, NUM_ROW_ANCHORS, NUM_COL_CELLS+1).to(device)
dummy_target = torch.randint(0, NUM_COL_CELLS+1, (4, NUM_LANES, NUM_ROW_ANCHORS)).to(device)
loss, l_cls, l_str = criterion(dummy_pred, dummy_target)
print(f'손실 테스트 — total: {loss.item():.4f}, cls: {l_cls:.4f}, str: {l_str:.4f}')

## 5. 학습 파이프라인

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = total_cls = total_str = 0.0
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        pred = model(imgs)
        loss, l_cls, l_str = criterion(pred, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        total_loss += loss.item()
        total_cls  += l_cls
        total_str  += l_str

    n = len(loader)
    return total_loss/n, total_cls/n, total_str/n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    TuSimple 평가 메트릭:
    - Accuracy: 예측 keypoint가 GT와 20px 이내인 비율
    - FP: 있다고 예측했지만 GT 없음
    - FN: GT는 있지만 예측 못 함
    """
    model.eval()
    val_loss = 0.0
    total_correct = total_valid = 0
    total_fp = total_fn = 0
    total_gt_present = total_pred_present = 0

    PIXEL_THRESH = 20   # 20px 이내 = 정답

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        pred = model(imgs)
        loss, _, _ = criterion(pred, labels)
        val_loss += loss.item()

        # argmax → 예측 column cell
        pred_col = pred.argmax(dim=-1)  # (B, L, R)

        B, L, R = pred_col.shape
        no_lane_idx = NUM_COL_CELLS

        # 픽셀 변환
        col_to_px = IMG_W / NUM_COL_CELLS

        for b in range(B):
            for l in range(L):
                for r in range(R):
                    gt  = labels[b, l, r].item()
                    pr  = pred_col[b, l, r].item()

                    gt_present  = (gt  != no_lane_idx)
                    pr_present  = (pr  != no_lane_idx)

                    if gt_present:
                        total_gt_present += 1
                        total_valid += 1
                        if pr_present:
                            diff_px = abs(pr - gt) * col_to_px
                            if diff_px <= PIXEL_THRESH:
                                total_correct += 1
                        else:
                            total_fn += 1   # FN: GT 있는데 예측 못 함
                    else:
                        if pr_present:
                            total_fp += 1   # FP: GT 없는데 있다고 예측
                        total_pred_present += 1 if pr_present else 0

    n = len(loader)
    accuracy = total_correct / total_valid if total_valid > 0 else 0.0
    fp_rate  = total_fp / (total_gt_present + 1e-9)
    fn_rate  = total_fn / (total_gt_present + 1e-9)

    return val_loss/n, accuracy, fp_rate, fn_rate


print('학습/평가 함수 준비 완료')

In [ ]:
# ── 학습 설정 ──────────────────────────────
EPOCHS    = 25
LR        = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {'train_loss':[], 'val_loss':[], 'accuracy':[], 'fp':[], 'fn':[]}
best_acc = 0.0

print(f'학습 시작 — {EPOCHS} epochs, LR={LR}, batch=16')
print(f'Train: {len(train_dataset)}장, Val: {len(val_dataset)}장\n')

for epoch in range(1, EPOCHS+1):
    train_loss, l_cls, l_str = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, acc, fp, fn    = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(acc)
    history['fp'].append(fp)
    history['fn'].append(fn)

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_ufld.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{EPOCHS} | '
              f'Train: {train_loss:.4f} (cls={l_cls:.4f}, str={l_str:.4f}) | '
              f'Val: {val_loss:.4f} | '
              f'Acc={acc:.4f} FP={fp:.4f} FN={fn:.4f}')

print(f'\n최고 Accuracy: {best_acc:.4f}')

## 6. 결과 시각화

In [ ]:
# ── 학습 곡선 ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='coral')
axes[0].set_title('손실 곡선')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['accuracy'], color='green', linewidth=2)
axes[1].set_title('Accuracy (TuSimple, 20px 기준)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=best_acc, color='green', linestyle='--', alpha=0.5, label=f'최고={best_acc:.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['fp'], label='FP', color='orange')
axes[2].plot(history['fn'], label='FN', color='red')
axes[2].set_title('FP / FN Rate')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('UFLD 학습 결과', fontsize=13)
plt.tight_layout()
plt.savefig('training_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 최고 모델로 정성적 결과 시각화 ─────────
model.load_state_dict(torch.load('best_ufld.pth', map_location=device))
model.eval()

COLORS = ['#FF4444', '#44FF44', '#4444FF', '#FFFF44']  # 차선별 색상

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

denorm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

col_to_px = IMG_W / NUM_COL_CELLS

for idx in range(8):
    ax = axes[idx]
    seed = 200000 + idx * 7   # 학습/검증에서 보지 않은 샘플
    img_orig, h_samples, gt_lanes = generate_lane_image(seed=seed)

    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    inp = transform(img_orig).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(inp)               # (1, 4, 56, 101)
        pred_col = pred.argmax(dim=-1)  # (1, 4, 56)

    ax.imshow(np.array(img_orig))

    # GT 차선 (점선)
    for li, lane in enumerate(gt_lanes):
        xs = [x for x, h in zip(lane, h_samples) if x != -2]
        ys = [h for x, h in zip(lane, h_samples) if x != -2]
        if xs:
            ax.plot(xs, ys, '--', color=COLORS[li], alpha=0.6, linewidth=1.5, label='GT' if li==0 else '')

    # 예측 차선 (실선)
    for li in range(NUM_LANES):
        pred_xs = []
        pred_ys = []
        for ri in range(NUM_ROW_ANCHORS):
            col = pred_col[0, li, ri].item()
            if col != NUM_COL_CELLS:   # no-lane 제외
                pred_xs.append(col * col_to_px)
                pred_ys.append(h_samples[ri])
        if pred_xs:
            ax.plot(pred_xs, pred_ys, '-', color=COLORS[li], linewidth=2.5,
                    label='Pred' if li==0 else '')

    ax.set_title(f'샘플 {idx+1}', fontsize=10)
    ax.axis('off')
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.suptitle('UFLD 정성적 결과 — GT(점선) vs 예측(실선)', fontsize=13)
plt.tight_layout()
plt.savefig('qualitative_results.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 구조적 손실(Structural Loss) 효과 분석

In [ ]:
# ── Ablation: lambda_str 효과 시각화 ──────
# 동일한 이미지에서 구조적 손실 유무에 따른 차이 시뮬레이션

@torch.no_grad()
def get_smooth_score(model, loader, device):
    """인접 row anchor 간 예측 변화량 (작을수록 부드러운 차선)"""
    model.eval()
    total_jitter = 0.0
    count = 0
    for imgs, _ in loader:
        imgs = imgs.to(device)
        pred = model(imgs)                     # (B,L,R,C+1)
        pred_col = pred.argmax(dim=-1).float() # (B,L,R)
        diff = (pred_col[:, :, 1:] - pred_col[:, :, :-1]).abs()
        total_jitter += diff.mean().item()
        count += 1
    return total_jitter / count

smooth = get_smooth_score(model, val_loader, device)
print(f'평균 row-anchor 간 열 변화량: {smooth:.2f} cells ({smooth * (IMG_W/NUM_COL_CELLS):.1f}px/row)')
print(f'→ 낮을수록 차선이 부드럽게 이어짐')

# 최종 지표 요약
_, final_acc, final_fp, final_fn = evaluate(model, val_loader, criterion, device)
print()
print('=' * 50)
print('UFLD 최종 성능 (합성 데이터, TuSimple 메트릭)')
print('=' * 50)
print(f'  Accuracy (20px): {final_acc:.4f} ({final_acc*100:.2f}%)')
print(f'  FP Rate:         {final_fp:.4f}')
print(f'  FN Rate:         {final_fn:.4f}')
print(f'  Smoothness:      {smooth:.2f} cells/row')
print('=' * 50)

## 8. Row-Anchor 방식 vs 세그멘테이션 방식 비교

In [ ]:
# ── 속도 벤치마크 ──────────────────────────
import time

model.eval()
dummy_batch = torch.randn(1, 3, IMG_H, IMG_W).to(device)

# Warm-up
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy_batch)
torch.cuda.synchronize()

# 측정
N_ITER = 200
start = time.perf_counter()
for _ in range(N_ITER):
    with torch.no_grad():
        _ = model(dummy_batch)
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

fps = N_ITER / elapsed
lat = elapsed / N_ITER * 1000
print(f'UFLD (ResNet-18, {IMG_H}x{IMG_W})')
print(f'  FPS:     {fps:.1f}')
print(f'  Latency: {lat:.2f} ms/frame')

# ── 접근 방식 비교 표 ──────────────────────
print()
methods = [
    ('Seg-based\n(FCN style)',      'Pixel-wise CE', '~30 FPS',  '높음',    'O(H×W×C) per px'),
    ('SCNN (2018)',                 'Seg + Message',  '~7 FPS',   '높음',    '슬라이딩 윈도우'),
    ('UFLD (2020)',                 'Row-anchor CLS', f'{fps:.0f} FPS', '중간',  'O(L×R) 분류만'),
    ('CLRNet (2022)',               'ROI + Cls',      '~200 FPS', '매우 높음','ROI 정밀 예측'),
]

print(f'{"방법":<22} {"출력 방식":<18} {"속도":<10} {"정확도":<10} {"특징"}')
print('-' * 80)
for m in methods:
    print(f'{m[0]:<22} {m[1]:<18} {m[2]:<10} {m[3]:<10} {m[4]}')

## 9. 면접 Q&A

**Q. UFLD의 핵심 아이디어를 설명해주세요.**

> 차선 검출을 픽셀 단위 세그멘테이션이 아니라 **행(row) 단위 분류 문제**로 재정의합니다. 이미지를 수평으로 56개 row anchor로 나누고, 각 row anchor에서 차선이 몇 번째 열(column cell)에 있는지 분류합니다. 세그멘테이션은 H×W개 픽셀 예측이 필요하지만, UFLD는 4(차선) × 56(row) = 224개 분류만 하면 됩니다. 덕분에 **300+ FPS**의 실시간 처리가 가능합니다.

**Q. 구조적 손실(Structural Loss)은 왜 필요한가요?**

> 분류 손실만 사용하면 인접한 row anchor 간 예측이 갑자기 튀는 현상(jitter)이 발생합니다. 실제 차선은 연속적이고 부드러운 곡선이므로, 인접 row anchor 간 예측 column 차이의 제곱합을 최소화하는 구조적 손실을 추가합니다. 이는 차선의 연속성이라는 기하학적 사전 지식을 손실 함수에 주입하는 방법입니다.

**Q. TuSimple 메트릭(Accuracy/FP/FN)을 설명해주세요.**

> - **Accuracy**: 예측한 keypoint와 GT keypoint의 픽셀 거리가 20px 이내인 비율. 전체 유효 keypoint 기준으로 계산합니다.
> - **FP (False Positive)**: GT 차선이 없는 row에서 차선이 있다고 예측한 비율.
> - **FN (False Negative)**: GT 차선이 있는 row에서 차선을 검출하지 못한 비율.
> 세 메트릭을 함께 봐야 단순 Accuracy 높이기 위해 항상 차선 없음으로 예측하는 trivial solution을 방지할 수 있습니다.

**Q. UFLD의 한계와 CLRNet이 어떻게 개선했나요?**

> UFLD는 column cell 해상도(100개)로 위치를 양자화하므로 세밀한 위치 정확도에 한계가 있습니다. 또한 급격한 커브 구간에서는 row-anchor 방식이 실패합니다. CLRNet(CVPR 2022)은 차선을 직선(ray)으로 모델링하고 ROI 기반 정밀 feature를 추출해 정확도를 크게 높이면서도 200+ FPS를 유지합니다.

In [ ]:
# ── 최종 결과 요약 ──────────────────────────
print('=' * 60)
print('skillup_round6 / 01 Lane Detection — 최종 결과')
print('=' * 60)
print(f'  모델:        UFLD (ResNet-18 + Row-Anchor Head)')
print(f'  데이터:      합성 차선 TuSimple 포맷 (train={len(train_dataset)}, val={len(val_dataset)})')
print(f'  Accuracy:    {final_acc:.4f} ({final_acc*100:.2f}%)')
print(f'  FP Rate:     {final_fp:.4f}')
print(f'  FN Rate:     {final_fn:.4f}')
print(f'  FPS:         {fps:.1f}')
print(f'  Latency:     {lat:.2f} ms')
print()
print('구현 핵심:')
print('  1. Row-anchor classification — 세그멘테이션 대비 18배 빠름')
print('  2. Structural Loss — 차선 연속성 제약 (사전 지식 주입)')
print('  3. TuSimple 메트릭 — Accuracy / FP / FN 파이프라인')
print('=' * 60)